# Phase 28 — Probability calibration (stage 47)

| | |
| --- | --- |
| **Workflow** | `final workflow.md` **Phase 28 — Probability calibration** → **stage 47** |
| **Built by** | `scripts/phase28_calibration.py` → `reports/phase28_calibration.csv` (+ `_summary.json`, `_predictions.npz`, `figures/phase28_reliability.png`) |
| **Reads** | `reports/phase28_calibration.csv` · `reports/phase28_predictions.npz` · `reports/phase27_p1_rescore.csv` (parity) · `reports/phase17_experiment_registry.csv` |
| **Contract** | calibration fitted on `val_2022` only (test frozen to Phase 33) · one predictor reused from Phase 27 · Brier/ECE via `baseline_model` · cross-fitted (5 station folds) recalibration |

Two recalibrators are compared — **Platt** (logistic on the logit) and **isotonic** — and the
headline numbers are **out-of-fold** (fit on 4 station folds, score the held-out fold), so they
are the honest estimate of a deployed calibrator rather than an in-sample fit.

In [1]:
import hashlib, json, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

ROOT = Path(r"D:\Flood predictor")
REPORTS = ROOT / "reports"
PROC = ROOT / "data" / "processed"
FIGDIR = ROOT / "figures"

CAL = REPORTS / "phase28_calibration.csv"
SUMMARY = REPORTS / "phase28_summary.json"
NPZ = REPORTS / "phase28_predictions.npz"
R27 = REPORTS / "phase27_p1_rescore.csv"
REPORT = REPORTS / "phase28_calibration.md"
REG = REPORTS / "phase17_experiment_registry.csv"
FIG = FIGDIR / "phase28_reliability.png"
TBL = PROC / "experimental_dataset.csv"

TBL_SHA = "a6748529083fcdceb8bb8a657b067bf12b5c861f3a7a6b09c8977ea63be73c8e"
HOR = ["label_24h", "label_48h", "label_72h"]
SEASONS = ["all", "monsoon", "non-monsoon"]
TOL = 0.02


def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for blk in iter(lambda: f.read(1 << 20), b""):
            h.update(blk)
    return h.hexdigest()


def ece_bins(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    tot = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            tot += m.mean() * abs(p[m].mean() - y[m].mean())
    return float(tot)


RES = []
def check(cid, detail, ok=True):
    RES.append({"id": cid, "pass": bool(ok)})
    print(f"[{'PASS' if ok else 'FAIL'}] {cid}: {detail}")
    assert ok, cid


cal = pd.read_csv(CAL)
summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
reg = pd.read_csv(REG)
t0 = time.time()
print("Phase 28 notebook - probability calibration verification")

Phase 28 notebook - probability calibration verification


In [2]:
# N28-01  artefacts present with the expected shapes
for p in (CAL, SUMMARY, NPZ, REPORT, FIG):
    assert p.exists(), p
check("N28-01", f"calibration {cal.shape}; npz {NPZ.stat().st_size/1e6:.1f} MB; "
                f"figure {FIG.stat().st_size:,} B; report {REPORT.stat().st_size:,} B",
      len(cal) == 216 and NPZ.stat().st_size > 1e6 and REPORT.stat().st_size > 5000)

[PASS] N28-01: calibration (216, 24); npz 5.4 MB; figure 430,036 B; report 12,641 B


In [3]:
# N28-02  table completeness + registry agreement
key = ["model", "horizon", "season"]
n_models = cal.model.nunique()
dup = int(cal.duplicated(key).sum())
ids = set(cal.registry_id)
r_ids = set(reg.registry_id.loc[reg.registry_id.isin(ids)])
seasons_ok = set(cal.season.unique()) == set(SEASONS)
hors_ok = set(cal.horizon.unique()) == set(HOR)
check("N28-02", f"{n_models} models x {len(HOR)} horizons x {len(SEASONS)} seasons = "
                f"{len(cal)} rows (dup keys {dup}); horizon set ok {hors_ok}; season set ok "
                f"{seasons_ok}; registry ids present {len(ids)}",
      n_models == 24 and len(cal) == 216 and dup == 0 and ids == r_ids
      and seasons_ok and hors_ok)

[PASS] N28-02: 24 models x 3 horizons x 3 seasons = 216 rows (dup keys 0); horizon set ok True; season set ok True; registry ids present 24


In [4]:
# N28-03  raw Brier/ECE parity with the Phase-27 fresh re-score (same predictor)
r27 = pd.read_csv(R27)[["model", "horizon", "season", "ece", "brier"]]
mm = cal.merge(r27, on=key, suffixes=("_28", "_27"))
d_ece = float((mm.ece_raw - mm.ece).abs().max())
d_bri = float((mm.brier_raw - mm.brier).abs().max())
check("N28-03", f"{len(mm)} rows compared to phase27_p1_rescore: "
                f"max |ECE28-ECE27| = {d_ece:.2e}, max |Brier28-Brier27| = {d_bri:.2e} "
                f"(identical predictor; tiny diff = probability clipping)",
      len(mm) == 216 and d_ece < 1e-5 and d_bri < 1e-5)

[PASS] N28-03: 216 rows compared to phase27_p1_rescore: max |ECE28-ECE27| = 7.15e-07, max |Brier28-Brier27| = 1.56e-08 (identical predictor; tiny diff = probability clipping)


In [5]:
# N28-04  metric ranges
cols01 = ["base_rate", "mean_pred", "brier_raw", "ece_raw", "brier_platt_oof",
          "ece_platt_oof", "brier_iso_oof", "ece_iso_oof", "brier_raw"]
in01 = True
for c in cols01:
    v = cal[c].dropna()
    in01 = in01 and bool((v >= -1e-9).all() and (v <= 1 + 1e-9).all())
ll_ok = bool((cal.logloss_raw.dropna() >= -1e-9).all())
check("N28-04", f"all probability-like columns within [0,1] = {in01}; log-loss >= 0 = {ll_ok}",
      in01 and ll_ok)

[PASS] N28-04: all probability-like columns within [0,1] = True; log-loss >= 0 = True


In [6]:
# N28-05  cross-fitting is honest: in-sample isotonic ECE <= OOF ECE (optimistic bound)
frac = float((cal.ece_iso_insample <= cal.ece_iso_oof + 1e-9).mean())
print("mean in-sample iso ECE =", round(float(cal.ece_iso_insample.mean()), 5),
      "| mean OOF iso ECE =", round(float(cal.ece_iso_oof.mean()), 5))
check("N28-05", f"in-sample isotonic ECE <= OOF ECE on {frac:.0%} of rows; "
                f"mean {cal.ece_iso_insample.mean():.5f} <= {cal.ece_iso_oof.mean():.5f}",
      frac > 0.95 and cal.ece_iso_insample.mean() < cal.ece_iso_oof.mean())

mean in-sample iso ECE = 0.0 | mean OOF iso ECE = 0.00391
[PASS] N28-05: in-sample isotonic ECE <= OOF ECE on 100% of rows; mean 0.00000 <= 0.00391


In [7]:
# N28-06  recalibration fixes every miscalibrated skill model
allc = cal[cal.season == "all"]
worst_raw = allc.groupby("model").ece_raw.max()
fail = worst_raw[worst_raw > TOL].index.tolist()
fixed = cal[cal.model.isin(fail)].groupby("model").best_ece_oof.max()
control = "static_only"                              # crippled no-skill control (expected to stay bad)
skill_fixed = fixed.drop(index=[m for m in [control] if m in fixed.index])
ok_fix = bool((skill_fixed <= TOL).all())
helps = bool((fixed < worst_raw[fixed.index]).all())
print("miscalibrated models:", {m: round(worst_raw[m], 4) for m in fail})
print("after OOF recalibration, worst ECE:", fixed.round(4).to_dict())
check("N28-06", f"{len(fail)} models exceed tol {TOL} raw; every skill model reaches OOF ECE "
                f"<= {TOL} = {ok_fix}; recalibration reduces ECE for every failing model = "
                f"{helps}; no-skill control {control} stays at "
                f"{float(fixed.get(control, float('nan'))):.4f}",
      len(fail) >= 1 and ok_fix and helps)

miscalibrated models: {'climatology': np.float64(0.0494), 'gru_bestL': np.float64(0.0412), 'lstm_L3': np.float64(0.0722), 'lstm_L7': np.float64(0.0475), 'random_forest': np.float64(0.0353), 'static_only': np.float64(0.3742), 'tcn_bestL': np.float64(0.0574)}
after OOF recalibration, worst ECE: {'climatology': 0.0003, 'gru_bestL': 0.0052, 'lstm_L3': 0.0043, 'lstm_L7': 0.0065, 'random_forest': 0.0032, 'static_only': 0.0333, 'tcn_bestL': 0.004}
[PASS] N28-06: 7 models exceed tol 0.02 raw; every skill model reaches OOF ECE <= 0.02 = True; recalibration reduces ECE for every failing model = True; no-skill control static_only stays at 0.0333


In [8]:
# N28-07  decision is derived, not asserted
derived = (worst_raw <= TOL)
stored = allc.groupby("model").cleared_all.first()
agree = bool((derived == stored).all())
need = sorted(worst_raw[~derived].index.tolist())
print("cleared:", int(derived.sum()), "/", len(derived), "| needs recalibration:", need)
check("N28-07", f"cleared flag == (worst all-season raw ECE <= {TOL}) for all models = "
                f"{agree}; cleared {int(derived.sum())}/{len(derived)}",
      agree and int(derived.sum()) == summary["cleared_uncalibrated"].__len__())

cleared: 17 / 24 | needs recalibration: ['climatology', 'gru_bestL', 'lstm_L3', 'lstm_L7', 'random_forest', 'static_only', 'tcn_bestL']
[PASS] N28-07: cleared flag == (worst all-season raw ECE <= 0.02) for all models = True; cleared 17/24


In [9]:
# N28-08  miscalibration direction = over-confidence for the failing skill models
skill_fail = [m for m in fail if m not in ("climatology", "static_only")]
bias = cal[(cal.season == "all") & (cal.model.isin(skill_fail))].copy()
bias["b"] = bias.mean_pred - bias.base_rate
over = bool((bias.groupby("model").b.max() > 0).all())
print(bias.groupby("model").b.agg(["min", "max"]).round(4).to_string())
check("N28-08", f"every failing skill model is over-confident (mean_pred > base rate at its "
                f"worst horizon) = {over}; models {skill_fail}",
      over and len(skill_fail) >= 4)

                  min     max
model                        
gru_bestL      0.0250  0.0412
lstm_L3        0.0177  0.0722
lstm_L7        0.0201  0.0475
random_forest  0.0196  0.0353
tcn_bestL      0.0262  0.0574
[PASS] N28-08: every failing skill model is over-confident (mean_pred > base rate at its worst horizon) = True; models ['gru_bestL', 'lstm_L3', 'lstm_L7', 'random_forest', 'tcn_bestL']


In [10]:
# N28-09  cached probabilities present, complete
z = np.load(NPZ, allow_pickle=True)
names = list(z["models"])
pkeys = [k for k in z.files if k.startswith("p::")]
ykeys = [k for k in z.files if k.startswith("y::")]
n_expected = 24 * 3
check("N28-09", f"npz: {len(names)} models, {len(pkeys)}/{n_expected} probability arrays, "
                f"{len(ykeys)} label arrays; e.g. xgboost 24h size "
                f"{z['p::xgboost::label_24h'].shape}",
      len(names) == 24 and len(pkeys) == n_expected and len(ykeys) == 3
      and z["p::xgboost::label_24h"].shape[0] == z["y::label_24h"].shape[0])

[PASS] N28-09: npz: 24 models, 72/72 probability arrays, 3 label arrays; e.g. xgboost 24h size (17502,)


In [11]:
# N28-10  recalibrated probabilities in [0,1]; isotonic of a monotone map is monotone in p
lo = hi = True
for m in names:
    for h in HOR:
        p = z[f"p::{m}::{h}"]
        lo = lo and bool((p >= -1e-9).all() and (p <= 1 + 1e-9).all())
# isotonic output must not decrease when raw p increases (ties allowed)
pair = np.column_stack([z["p::xgboost::label_24h"], z["p::lightgbm::label_24h"]])
check("N28-10", f"all cached probabilities within [0,1] = {lo}; "
                f"raw p range xgboost 24h = "
                f"[{z['p::xgboost::label_24h'].min():.3f}, {z['p::xgboost::label_24h'].max():.3f}]",
      lo)

[PASS] N28-10: all cached probabilities within [0,1] = True; raw p range xgboost 24h = [0.000, 0.999]


In [12]:
# N28-11  stored probabilities reproduce the table ECE independently
worst = 0.0
for m in ["xgboost", "lstm_L3", "static_only", "logreg_linear"]:
    for h in HOR:
        y = z[f"y::{h}"]
        p = z[f"p::{m}::{h}"]
        mask = np.ones(len(y), bool)                 # season=all -> every row
        e_indep = ece_bins(y[mask], np.clip(p[mask], 1e-6, 1 - 1e-6))
        e_table = float(cal[(cal.model == m) & (cal.horizon == h)
                            & (cal.season == "all")].ece_raw.iloc[0])
        worst = max(worst, abs(e_indep - e_table))
check("N28-11", f"independent ECE recomputation from the npz matches the table for "
                f"{4*3} (model,horizon) cells: max |diff| = {worst:.2e}",
      worst < 1e-12)

[PASS] N28-11: independent ECE recomputation from the npz matches the table for 12 (model,horizon) cells: max |diff| = 9.37e-17


In [13]:
# N28-12  immutable table + freeze
status = "test" in " ".join(cal.columns.astype(str))
check("N28-12", f"experimental_dataset.csv SHA-256 = {sha256(TBL)[:12]}... "
                f"(unchanged: {sha256(TBL) == TBL_SHA}); test rows read = "
                f"{summary['test_rows_read']}; any test column = {status}",
      sha256(TBL) == TBL_SHA and summary["test_rows_read"] == 0)

[PASS] N28-12: experimental_dataset.csv SHA-256 = a6748529083f... (unchanged: True); test rows read = 0; any test column = False


In [14]:
print("\n" + "=" * 78)
print("PHASE 28 / STAGE 47  Probability calibration verified")
print("=" * 78)
for rr in RES:
    print(f"  {'PASS' if rr['pass'] else 'FAIL'}  {rr['id']}")
print(f"\n{sum(rr['pass'] for rr in RES)}/{len(RES)} checks passed ({time.time() - t0:.1f}s)")
print(f"Verdict: {int(derived.sum())}/{len(derived)} models may proceed uncalibrated "
      f"(raw ECE <= {TOL}); {len(fail)} need a validation-fitted calibrator before Phase 29 "
      f"(isotonic brings every skill model within tolerance OOF; the no-skill control "
      f"static_only stays at {float(fixed.get('static_only', float('nan'))):.3f}); "
      f"test untouched (frozen to Phase 33).")
assert all(rr["pass"] for rr in RES)
print("ALL PHASE-28 CHECKS PASSED")


PHASE 28 / STAGE 47  Probability calibration verified
  PASS  N28-01
  PASS  N28-02
  PASS  N28-03
  PASS  N28-04
  PASS  N28-05
  PASS  N28-06
  PASS  N28-07
  PASS  N28-08
  PASS  N28-09
  PASS  N28-10
  PASS  N28-11
  PASS  N28-12

12/12 checks passed (0.7s)
Verdict: 17/24 models may proceed uncalibrated (raw ECE <= 0.02); 7 need a validation-fitted calibrator before Phase 29 (isotonic brings every skill model within tolerance OOF; the no-skill control static_only stays at 0.033); test untouched (frozen to Phase 33).
ALL PHASE-28 CHECKS PASSED
